# Memory allocation analysis

This notebook analyses the **memory allocation timeline** from the `profile_memory=True` trace
(Run B) produced by `train.py`. It covers Part B of Exercise 4.

See `analysis.ipynb` for kernel timing, bandwidth, Perfetto trace view, and operator stats (Sections 1–5).

**Prerequisites**
```bash
pip install HolisticTraceAnalysis plotly
```

**Trace location** — copy the memory trace from the cluster first:
```bash
rsync -r <host>:~/<path>/4_profile/profiler_output_memory ./
```

In [ ]:
import glob, json, os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.offline as pyo
pyo.init_notebook_mode() # Set notebook mode to work in offline

TRACE_DIR_B = os.path.abspath("./profiler_output_memory")  # Run B: memory allocation timeline


def load_trace_files(trace_dir: str) -> dict[int, str]:
    """Return a Dict[rank, absolute_path] for all .pt.trace.json files under trace_dir."""
    files = sorted(glob.glob(os.path.join(trace_dir, "**", "*.pt.trace.json"), recursive=True))
    if not files:
        raise FileNotFoundError(
            f"No .pt.trace.json files found under {trace_dir!r}.\n"
            "Copy the memory trace from the cluster first:\n"
            "  rsync -r <host>:~/<path>/4_profile/profiler_output_memory ./"
        )
    result = {}
    for fp in files:
        rank_dir = os.path.basename(os.path.dirname(fp))  # e.g. 'rank_0'
        rank = int(rank_dir.split("_")[-1]) if rank_dir.startswith("rank_") else len(result)
        result[rank] = fp
    return result


trace_files_b = load_trace_files(TRACE_DIR_B)

print("Run B — memory analysis traces:")
for rank, fp in sorted(trace_files_b.items()):
    print(f"  rank {rank}: {fp}")

---
## Peak usage and allocation timeline

Parsed from the `[memory]` instant events that `torch.profiler` writes when `profile_memory=True`.
Each event captures a snapshot of the allocator state at the moment of every allocation or free,
giving a complete picture of memory pressure over the profiling window.

- **`allocated_gb`** — bytes actively held by live tensors
- **`reserved_gb`** — bytes held by the caching allocator (always ≥ allocated; the gap is fragmentation headroom pre-reserved to avoid repeated `cudaMalloc` calls)

> **Note:** GPU kernel timings in this trace are inflated because `profile_memory=True` adds CPU
> overhead around every op. Use `analysis.ipynb` (Run A trace) for accurate kernel timing.

In [ ]:
with open(trace_files_b[0]) as f:
    trace = json.load(f)

t0 = trace["traceEvents"][0]["ts"]

mem_events = [
    e for e in trace["traceEvents"]
    if e.get("ph") == "i" and e.get("name") == "[memory]"
    and e.get("args", {}).get("Device Type") == 1
]
mem_df = pd.DataFrame([
    {
        "ts_ms":        (e["ts"] - t0) / 1e6,
        "allocated_gb": e["args"]["Total Allocated"] / 1e9,
        "reserved_gb":  e["args"]["Total Reserved"]  / 1e9,
    }
    for e in mem_events
]).sort_values("ts_ms")

peak_allocated = mem_df["allocated_gb"].max()
peak_reserved  = mem_df["reserved_gb"].max()
print(f"Peak allocated : {peak_allocated:.2f} GB")
print(f"Peak reserved  : {peak_reserved:.2f} GB")

raw_steps = [e for e in trace["traceEvents"]
             if e.get("ph") == "X" and str(e.get("name", "")).startswith("ProfilerStep")]
steps_by_name: dict = {}
for e in raw_steps:
    name = e["name"]
    if name not in steps_by_name or e["dur"] > steps_by_name[name]["dur"]:
        steps_by_name[name] = e
steps = sorted(steps_by_name.values(), key=lambda e: e["ts"])

fig = px.line(mem_df, x="ts_ms", y=["allocated_gb", "reserved_gb"],
    title="GPU memory over time (shaded = ProfilerStep)",
    labels={"ts_ms": "Time (ms)", "value": "Memory (GB)", "variable": ""})

colors = ["rgba(255,165,0,0.15)", "rgba(0,128,255,0.15)", "rgba(0,200,100,0.15)"]
for i, step in enumerate(steps):
    start_ms = (step["ts"] - t0) / 1e6
    end_ms   = (step["ts"] + step["dur"] - t0) / 1e6
    fig.add_vrect(x0=start_ms, x1=end_ms, fillcolor=colors[i % len(colors)],
        layer="below", line_width=0,
        annotation_text=step["name"], annotation_position="top left",
        annotation_font_size=10)
fig.show()

### Questions to consider

<details>
<summary><b>1. How much headroom is there before hitting OOM?</b></summary>

Compare `peak_allocated` against your GPU's total VRAM. A large gap means batch size, sequence length, or model size can be increased before hitting OOM. A small gap means you are close to the limit — consider gradient checkpointing or mixed precision to reduce activation memory.
</details>

<details>
<summary><b>2. Why is reserved memory much larger than allocated memory?</b></summary>

The PyTorch caching allocator pre-reserves memory in large blocks to avoid repeated `cudaMalloc` calls. A large `reserved − allocated` gap is normal and does not indicate a leak. Reserved memory only returns to the OS when `torch.cuda.empty_cache()` is called explicitly.
</details>

<details>
<summary><b>3. Are memory spikes consistent across steps?</b></summary>

In the allocation timeline, each `ProfilerStep` shaded region should show the same spike pattern: rise during forward pass, peak during backward pass, drop after the optimizer step frees activations. An unusually tall spike in one step suggests an unexpected large intermediate tensor — check for `retain_graph=True` or an unintended in-place operation keeping activations alive.
</details>